# RadCluster_2_1 — Digital-Twin Campaign Control (T3 rev 6)

**Run this notebook unchanged on every participating machine. Nothing in it
is machine-specific and nothing needs editing.**

The host identifies itself from `machines.json`, which is also the single
source of the frozen grid — the notebook *builds its command line from that
file* rather than carrying its own copy, so the grid cannot drift between
machines and is never retyped. `git pull` in section 1 is the only way
settings change.

Run sections in order. **0–4 are pre-flight and must all pass before 5
launches the real run.**


## 0 — Setup


In [1]:
import json, subprocess, sys, time, collections
from pathlib import Path

def _find_root():
    """Locate digital_twin from wherever the kernel happens to start."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base, base / 'RadCluster_2_1' / 'digital_twin'):
            if (cand / 'run_ensemble.py').exists() and (cand / 'machines.json').exists():
                return cand
    raise SystemExit('cannot locate RadCluster_2_1/digital_twin from ' + str(Path.cwd()))

HERE = _find_root()
sys.path.insert(0, str(HERE))
import campaign_ops as ops, run_ensemble as RE

REPO     = Path(subprocess.run(['git','rev-parse','--show-toplevel'], cwd=HERE,
                               capture_output=True, text=True).stdout.strip())
REGISTRY = HERE / 'machines.json'
RESULTS  = HERE / 'results'
PY       = sys.executable
print('repo :', REPO)
print('here :', HERE)


repo : /Users/ghoni/Documents/GitHub/RadCluster
here : /Users/ghoni/Documents/GitHub/RadCluster/RadCluster_2_1/digital_twin


## 1 — Pull

Every participant must run the same code, design and grid. `merge_and_sobol`
compares `git_sha`, `solver_sha256`, `workbook_sha256`, `design_sha256`,
`run_cfg_sha`, `weights_sha` and `of`, and reports a PROVENANCE SPLIT on any
disagreement. **Re-run this cell whenever settings change — that is the only
step needed to pick up a new grid.**


In [2]:
print(subprocess.run(['git','pull','--ff-only'], cwd=REPO,
                     capture_output=True, text=True).stdout.strip())
print('HEAD =', subprocess.run(['git','rev-parse','--short','HEAD'], cwd=REPO,
                               capture_output=True, text=True).stdout.strip())


Already up to date.
HEAD = 762c969


## 2 — Read the frozen campaign settings

Everything below is derived from `machines.json`. Nothing is hard-coded here.


In [3]:
reg    = json.loads(REGISTRY.read_text())
G      = reg['grid']
DESIGN = HERE / reg['design']
EXPECTED_SHA = reg['run_cfg_sha']
TIMEOUT_S    = reg['timeout_s']

GRID = ['--equations', G['equations'],
        '--I', str(G['I']), '--V', str(G['V']),
        '--i-discrete', str(G['i_discrete']), '--v-discrete', str(G['v_discrete']),
        '--i-bin', str(G['i_bin']), '--v-bin', str(G['v_bin']),
        '--shape-function', G['shape_function'],
        '--i-mobile-default', str(G['i_mobile_default']),
        '--v-mobile-default', str(G['v_mobile_default']),
        '--dose', str(G['dose']), '--rtol', str(G['rtol']),
        '--solver-mode', G['solver_mode']]

print('design      ', DESIGN.name)
print('grid        ', ' '.join(f'{k}={v}' for k, v in G.items()))
print('run_cfg_sha ', EXPECTED_SHA)
print('timeout_s   ', TIMEOUT_S)
print('\nNOTE: i_mobile is PHYSICS, not a numerical knob. These rows cannot be')
print('pooled with any run at a different i_mobile; run_cfg_sha enforces it.')


design       T3_rev6.csv
grid         I=10000 V=5000 dose=1.0 equations=bin_moment solver_mode=active_window rtol=1e-06 i_discrete=40 v_discrete=5 i_bin=20 v_bin=20 shape_function=linear i_mobile_default=40 v_mobile_default=5
run_cfg_sha  482d59556632dff8
timeout_s    12000

NOTE: i_mobile is PHYSICS, not a numerical knob. These rows cannot be
pooled with any run at a different i_mobile; run_cfg_sha enforces it.


## 3 — Who am I?

Detection is by host fingerprint and **fails loudly** on no match or an
ambiguous match. A wrong index means two machines compute the same rows and
some rows are computed by nobody — which surfaces at merge time as *rows
MISSING*, indistinguishable from a machine that never reported.

If it refuses: `RE.register_host(<index>, REGISTRY)`, then commit and push
`machines.json`. Do not guess an index.


In [4]:
me    = RE.detect_machine(reg, RE._host_facts())   # raises if unrecognised
W     = [float(w) for w in reg['weights'].split(',')]
rows, meta = RE.read_design(DESIGN)
mine  = [r for r in rows if RE.assign_machine(int(r['row_id']), reg['of'], W) == me['index']]
print(f"machine {me['index']} = {me['name']}   slots {me['slots']}   "
      f"speed {me['speed']} ({me['speed_source']})   weight {me['weight']}")
print(f'owns {len(mine)} of {len(rows)} rows')


machine 3 = MacBook Air   slots 8   speed 0.85 (DECLARED)   weight 6.8
owns 233 of 1008 rows


## 4 — Build, agreement gate, and a bounded PRE-FLIGHT

**Do not skip the pre-flight.** Its absence cost two aborted Hoffman2
submissions: every row failed in 5–8 s with `d100=nan` while the provenance
line looked perfect — that line prints *before* any row runs and proves only
that the config was assembled.

The cause is unresolved and **may affect any Linux host**: rows succeed
through a direct call but fail through `run_ensemble`'s multiprocessing pool
at the production grid. So it must run *on this machine*.


In [5]:
info = ops.ensure_solver()
r = subprocess.run([PY, str(HERE / 'check_machine.py')], cwd=HERE,
                   capture_output=True, text=True)
print(r.stdout[-1800:])
assert r.returncode == 0, 'AGREEMENT GATE FAILED — do not contribute rows from this build'


  solver: OK  /Users/ghoni/Documents/GitHub/RadCluster/RadCluster_2_1/build/solver  sha 9366e57649416372
machine   Mac.san.rr.com  (macOS-15.7.4-arm64-arm-64bit)
python    3.9.6
git       762c969c16bc
solver    9366e57649416372  exists=True
workbook  9253e7a0370af966

running probe (I=150, 0.02 dpa, ~30 s) ...

  reference generated on Nasr-Workstation (git 76efc2a46f6c, solver c28893e7d4a119e2)
  note: git SHA differs from the reference machine - pull first if that is not intentional.

  field                    this machine        reference    rel diff
  Di_eff                7.085348836e-12  7.085348836e-12    0.00e+00
  Dv_eff                2.125201773e-13  2.125201773e-13    0.00e+00
  conv_psuccess         2.181172484e-06  2.181172484e-06    0.00e+00
  conv_psuccess_abs     1.000000000e+00  1.000000000e+00    0.00e+00
  N_loops_100           1.679711672e+20  1.679711678e+20    3.62e-09
  N_loops_111           2.904994897e+23  2.904994914e+23    5.81e-09
  mean_n_100            2

In [ ]:
import re as _re
t0 = time.time()
pf = RESULTS / '_preflight.jsonl'
r  = subprocess.run([PY, '-u', str(HERE / 'run_ensemble.py'),
                     '--design', str(DESIGN), '--machine', 'auto', *GRID,
                     '--timeout-s', str(TIMEOUT_S),
                     '--limit', '2', '--workers', '2', '--out', str(pf)],
                    cwd=HERE, capture_output=True, text=True)
out = r.stdout + r.stderr
print(out[-1800:])
m   = _re.search(r'"run_cfg_sha": "([0-9a-f]+)"', out)
sha = m.group(1) if m else None
print(f'\n  run_cfg_sha {sha}  (expected {EXPECTED_SHA})')
print(f'  FAIL lines  {out.count("FAIL")}')
print(f'  elapsed     {time.time()-t0:.0f} s   -> per-row cost sets the ETA')
assert sha == EXPECTED_SHA, 'GRID MISMATCH — this machine is not on the frozen grid'
assert out.count('FAIL') == 0, 'ROWS FAILED — do NOT launch; report the error text'
for p in RESULTS.glob('_preflight*'): p.unlink()
print('\n  PRE-FLIGHT PASSED')


## 5 — Launch

Detached, so the notebook can be closed. Resumption is the design: re-running
this cell skips `row_id`s already present and only ever adds.

> `--timeout-s` is a budget, not a cap. On expiry the solver is asked to
> finalize and its **partial trajectory is kept**, contributing to every rung
> of the dose ladder it reached. Note it bounds a *segment* subprocess, not a
> whole row (measured 2026-08-06: 20 503 s rows completed under 12 000 s).


In [ ]:
ops.clear_stop()
LOG = RESULTS / f"worker_machine{me['index']}.log"
cmd = [PY, '-u', str(HERE / 'run_ensemble.py'), '--design', str(DESIGN),
       '--machine', 'auto', *GRID, '--timeout-s', str(TIMEOUT_S)]
print(' '.join(cmd), '\n')
with open(LOG, 'w') as fh:
    proc = subprocess.Popen(cmd, cwd=HERE, stdout=fh, stderr=subprocess.STDOUT,
                            start_new_session=True)
print(f'launched pid {proc.pid} -> {LOG.name}')
time.sleep(40); print(open(LOG).read()[:1200])


## 6 — Monitor


In [ ]:
ops.watch(DESIGN, RESULTS, n_machines=reg['of'], interval=120)   # Ctrl-C to stop watching


In [ ]:
st = ops.campaign_status(DESIGN, RESULTS, n_machines=reg['of'])
ops.render_status(st, ops.load_targets())


### 6b — Throughput and dose-ladder coverage

Throughput, not wall-time-per-row, is the measure that survives concurrency —
planning from per-row wall measured at low concurrency is what produced a
3780 s estimate against a 17 828 s reality.


In [ ]:
recs = ops.load_results(RESULTS)
walls = [r['wall_s'] for r in recs.values() if r.get('wall_s') and not r.get('solver_rc')]
if walls:
    import statistics
    mean = statistics.mean(walls)
    cap  = sum(float(w) for w in reg['weights'].split(','))
    print(f'rows done {len(walls)}   mean wall {mean:.0f} s   '
          f"throughput {me['slots']*3600/mean:.2f} rows/h on this machine")
    print(f'projected campaign: {len(rows)*mean/cap/3600:.0f} h over {cap:.1f} slot-equivalents')
cov = collections.Counter()
for r in recs.values():
    for rung in (r.get('at_dose') or {}): cov[rung] += 1
for k in sorted(cov, key=float): print(f'  {k:>4s} dpa : {cov[k]:4d} rows')


## 7 — Graceful stop / resume

Rows in flight finish and are written; no new rows start.


In [ ]:
ops.request_stop('put the real reason here')


In [ ]:
ops.clear_stop()


## 8 — Pool and report (one machine, after everyone has pushed)

Push `results/*.jsonl` **and** `*.manifest.json` — the manifest is what sizes
the next campaign from measured throughput.

Run it **both ways**. If the parameter *ranking* is unchanged with and without
`--require-converged`, truncation did not buy it anything — the empirical form
of the claim the no-gating policy rests on.


In [ ]:
for extra in ([], ['--require-converged']):
    print('='*70); print('  merge_and_sobol', *extra); print('='*70)
    r = subprocess.run([PY, str(HERE / 'merge_and_sobol.py'),
                        '--design', str(DESIGN), '--results', str(RESULTS),
                        '--at-dose', str(reg['grid']['dose']), *extra],
                       cwd=HERE, capture_output=True, text=True)
    print(r.stdout[-4000:])
